# 08 - Pipeline

## Proyecto
Detección de Sitios Web Fraudulentos (Phishing)

## Objetivo de esta notebook
Integrar todas las transformaciones del Sprint 2 en un `sklearn.Pipeline` reproducible:
- encapsular limpieza, encoding y escalado en `ColumnTransformer`
- validar que el pipeline aplica `fit` solo sobre train y `transform` sobre test
- persistir el pipeline con `joblib` para reutilizarlo en los Sprints 3 y 4
- versionar el dataset procesado

Esta notebook corresponde al rol de **Pipeline Builder (Sprint 2)**.

## Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

## Carga del dataset con features (post-feature engineering)

In [2]:
df = pd.read_csv("../data/interim/dataset_features.csv")
print(f"Dimensiones: {df.shape}")
df.head()

Dimensiones: (5849, 38)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,Links_pointing_to_page,Statistical_report,Result,url_risk_score,security_score,total_suspicious_count,total_legitimate_count,net_signal_ratio,ssl_traffic_interaction,content_risk_score
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,-1,-1,4,-4,16,13,-3,1,3
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,1,-2,8,18,10,0,2
2,1,0,1,1,1,-1,-1,-1,-1,1,...,0,-1,-1,2,-2,12,14,2,-1,3
3,1,0,1,1,1,-1,-1,-1,1,1,...,-1,1,-1,2,-2,10,16,6,-1,2
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,1,1,2,0,9,16,7,0,3


## Separación de features y target

In [3]:
X = df.drop(columns=["Result"])
y = df["Result"]

# Clasificación de columnas
# Todas son numéricas (ternarias originales + nuevas features derivadas)
numeric_cols = X.columns.tolist()

print(f"Total de features: {len(numeric_cols)}")
print(f"Features originales (ternarias): {len([c for c in numeric_cols if c not in ['url_risk_score','security_score','total_suspicious_count','total_legitimate_count','net_signal_ratio','ssl_traffic_interaction','content_risk_score']])}")
print(f"Features derivadas (nuevas): {len([c for c in numeric_cols if c in ['url_risk_score','security_score','total_suspicious_count','total_legitimate_count','net_signal_ratio','ssl_traffic_interaction','content_risk_score']])}")  

Total de features: 37
Features originales (ternarias): 30
Features derivadas (nuevas): 7


## Train/Test Split con estratificación

Se realiza el split antes de cualquier `fit` del pipeline para evitar data leakage.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance train: {dict(y_train.value_counts().sort_index())}")
print(f"Balance test:  {dict(y_test.value_counts().sort_index())}")

Train: (4679, 37) | Test: (1170, 37)
Balance train: {-1: np.int64(2415), 1: np.int64(2264)}
Balance test:  {-1: np.int64(604), 1: np.int64(566)}


## Construcción del Pipeline de preprocesamiento

Se construye un `ColumnTransformer` que aplica un pipeline numérico a todas las columnas:
- `SimpleImputer` con estrategia de mediana (precaución ante eventuales nulos en datos nuevos)
- `StandardScaler` para normalizar la escala

Este pipeline puede extenderse en Sprints futuros añadiendo más transformadores sin romper la estructura.

In [5]:
# Sub-pipeline para variables numéricas
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),  # Precaución ante nulos en producción
    ("scaler", StandardScaler())                     # Escalado estándar
])

# ColumnTransformer: aplica el pipeline numérico a todas las columnas
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols)
    ],
    remainder="drop"  # Eliminar columnas no especificadas
)

print("Pipeline de preprocesamiento construido:")
print(preprocessor)

Pipeline de preprocesamiento construido:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['having_IP_Address', 'URL_Length',
                                  'Shortining_Service', 'having_At_Symbol',
                                  'double_slash_redirecting', 'Prefix_Suffix',
                                  'having_Sub_Domain', 'SSLfinal_State',
                                  'Domain_registeration_length', 'Favicon',
                                  'port', 'HTTPS_token', 'Request_URL',
                                  'URL_of_Anchor', 'Links_in_tags', 'SFH',
                                  'Submitting_to_email', 'Abnormal_URL',
                                  'Redirect', 'on_mouseover', 'RightClick',
                       

## Entrenamiento del pipeline exclusivamente sobre el train set

El `fit` aprende los parámetros (media, desviación estándar, mediana) **solo** a partir de `X_train`. Luego se aplica `transform` sobre ambos conjuntos.

In [6]:
# fit SOLO en train
preprocessor.fit(X_train)

# transform en train y test por separado
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train transformado: {X_train_transformed.shape}")
print(f"X_test transformado:  {X_test_transformed.shape}")

# Verificación: media ≈ 0 y std ≈ 1 en train (tras StandardScaler)
X_train_df = pd.DataFrame(X_train_transformed, columns=numeric_cols)
print(f"\nMedia promedio en train (debe ser ~0): {X_train_df.mean().mean():.6f}")
print(f"Std promedio en train (debe ser ~1):   {X_train_df.std().mean():.6f}")

X_train transformado: (4679, 37)
X_test transformado:  (1170, 37)

Media promedio en train (debe ser ~0): 0.000000
Std promedio en train (debe ser ~1):   1.000107


## Verificación de no data leakage en el test set

La media y std en test no serán exactamente 0 y 1 porque se usaron los parámetros aprendidos en train. Esto es el comportamiento correcto.

In [7]:
X_test_df = pd.DataFrame(X_test_transformed, columns=numeric_cols)
print(f"Media promedio en test (no necesariamente ~0): {X_test_df.mean().mean():.6f}")
print(f"Std promedio en test (no necesariamente ~1):   {X_test_df.std().mean():.6f}")
print("\n✓ El test fue transformado con parámetros aprendidos en train (correcto, sin leakage).")

Media promedio en test (no necesariamente ~0): 0.011338
Std promedio en test (no necesariamente ~1):   0.990354

✓ El test fue transformado con parámetros aprendidos en train (correcto, sin leakage).


## Persistencia del pipeline con joblib

El pipeline se serializa para reutilizarlo en los Sprints 3, 4 y 6 sin necesidad de reentrenar las transformaciones.

In [8]:
os.makedirs("../models", exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessor.pkl")
print("Pipeline guardado en ../models/preprocessor.pkl")

# Verificación: cargar y aplicar
preprocessor_loaded = joblib.load("../models/preprocessor.pkl")
X_check = preprocessor_loaded.transform(X_test)
print(f"\nVerificación de carga: transform aplicado correctamente. Shape: {X_check.shape}")
print("✓ Pipeline serializado y verificado exitosamente.")

Pipeline guardado en ../models/preprocessor.pkl

Verificación de carga: transform aplicado correctamente. Shape: (1170, 37)
✓ Pipeline serializado y verificado exitosamente.


## Guardado de los conjuntos transformados

Se guardan los conjuntos de entrenamiento y test listos para el Sprint 3 (Modeling).

In [9]:
os.makedirs("../data/processed", exist_ok=True)

# Guardar X_train y X_test transformados
X_train_final = pd.DataFrame(X_train_transformed, columns=numeric_cols)
X_train_final["Result"] = y_train.values
X_train_final.to_csv("../data/processed/train_preprocessed.csv", index=False)

X_test_final = pd.DataFrame(X_test_transformed, columns=numeric_cols)
X_test_final["Result"] = y_test.values
X_test_final.to_csv("../data/processed/test_preprocessed.csv", index=False)

print("Archivos guardados:")
print(f"  ../data/processed/train_preprocessed.csv : {X_train_final.shape}")
print(f"  ../data/processed/test_preprocessed.csv  : {X_test_final.shape}")

Archivos guardados:
  ../data/processed/train_preprocessed.csv : (4679, 38)
  ../data/processed/test_preprocessed.csv  : (1170, 38)


## Validación end-to-end del pipeline

Se verifica que el pipeline completo puede aplicarse sobre datos nuevos (simulando un escenario de producción).

In [10]:
# Simular 3 observaciones nuevas (con las mismas columnas que X)
nueva_obs = X_test.iloc[:3].copy()

# Cargar el pipeline desde disco y aplicar transform
pipe_loaded = joblib.load("../models/preprocessor.pkl")
resultado = pipe_loaded.transform(nueva_obs)

print("Validación end-to-end:")
print(f"  Input shape:  {nueva_obs.shape}")
print(f"  Output shape: {resultado.shape}")
print("\n✓ El pipeline cargado desde disco transforma correctamente nuevas observaciones.")
print("\n✓ Listo para el Sprint 3: Modeling.")

Validación end-to-end:
  Input shape:  (3, 37)
  Output shape: (3, 37)

✓ El pipeline cargado desde disco transforma correctamente nuevas observaciones.

✓ Listo para el Sprint 3: Modeling.


## Diagrama del pipeline

```
Datos crudos (dataset.csv)
       |
       ▼
[05_data_cleaning.ipynb] → Eliminar duplicados
       |
       ▼
[06_feature_eng.ipynb]  → Crear 7 features derivadas
       |
       ▼
train_test_split(stratify=y, test_size=0.20, random_state=42)
       |
       ├──────────────────────────────┐
       ▼                              ▼
   X_train                        X_test
       |                              |
       ▼                              |
[07_class_balance.ipynb]              |
SMOTE → X_train_bal               (sin modificar)
       |                              |
       └──────────────┬───────────────┘
                      ▼
          [08_pipeline.ipynb]
          ColumnTransformer:
            - SimpleImputer(median)
            - StandardScaler()
          fit(X_train) → transform(X_train, X_test)
                      |
                      ▼
          preprocessor.pkl (joblib)
          data/processed/train_preprocessed.csv
          data/processed/test_preprocessed.csv
                      |
                      ▼
          [Sprint 3: Modeling]
```

## Resumen del Sprint 2 — Pipeline Builder

En esta notebook se integraron todas las transformaciones del Sprint 2 en un pipeline reproducible:

| Componente | Detalle |
|---|---|
| `SimpleImputer` | Estrategia: mediana (robustez ante nuevos datos en producción) |
| `StandardScaler` | Normaliza media=0, std=1 (beneficio para modelos lineales y KNN) |
| `ColumnTransformer` | Aplica el pipeline numérico a todas las features |
| `fit` | Aplicado exclusivamente sobre X_train (sin data leakage) |
| `transform` | Aplicado sobre X_train y X_test con los parámetros de train |
| `joblib.dump` | Pipeline serializado en `models/preprocessor.pkl` |
| Archivos generados | `data/processed/train_preprocessed.csv`, `data/processed/test_preprocessed.csv` |

El pipeline está listo para ser reutilizado en el **Sprint 3: Modeling (baseline)**.